In [1]:
import os
import sys
sys.path.append('..')

from src.vectorstore import csv_loader, build_vectorstore
from src.rag_pipeline import load_llm, semantic_retriever, initialize_rag_chain, invoke_rag_chain
from src.hybrid import preprocess_query, bm25_retriever, hybrid_retriever
from src.prompts import prompt

c:\Users\liauw\Desktop\Sputnik\2025-26\Courses\block-6\575-nlp\DSCI_575_project_cliauwyt_cea\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Define model

In [3]:
llm = load_llm()

# RAG Semantic

## Load vector store

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

corpus_path = '../data/processed/preprocessed_corpus.csv'
vector_path = "../data/processed/vector_store"
docs = csv_loader(corpus_path)

if not os.path.exists(vector_path):
    build_vectorstore(docs, vector_path, embeddings)
    print(f"Saved vector store to {vector_path}")
    
vectorstore = FAISS.load_local(
    vector_path, embeddings, allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval

In [5]:
vector_retriever = semantic_retriever(vectorstore)
query = "what is the best soap"
vector_retriever.invoke(query)

[Document(id='39aa2e66-1341-4f44-8bec-a3f258152067', metadata={'source': 'data/processed/preprocessed_corpus.csv', 'row': 6054, 'asin': 'B0716PQVP2', 'product_title': 'Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool', 'rating': '5.0', 'review_text': 'Must have with bars of soap ! You will love !'}, page_content='text: double layer exfoliating mesh soap saver pouch bubble foam net handmade soap mesh bag body facial cleaning tool health personal care great bar soap love'),
 Document(id='b3cffd8e-54ae-413b-8758-c607965b7230', metadata={'source': 'data/processed/preprocessed_corpus.csv', 'row': 5693, 'asin': 'B08DV37PZV', 'product_title': 'Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack', 'rating': '5.0', 'review_text': 'That "blue" dish soap is more difficult to rinse off.  I like Palmolive because it cleans well and rinses off easily.'}, page_content='text: palmolive ultra original dish liquid pack 

## Pipeline

In [6]:
rag_chain = initialize_rag_chain(vector_retriever, llm, prompt)
print(invoke_rag_chain(rag_chain, query))

Based on the Amazon reviews, it seems that a bar soap is a popular choice. The reviewer for product ASIN: B0716PQVP2 highly recommends the "Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch" which is likely used with bar soap.


# Hybrid RAG: Semantic Search + BM25

### BM25 Retriever

In [7]:
keyword_retriever = bm25_retriever(docs)

# preprocess query so it's consistent with corpus preprocessing
query = preprocess_query(query)

# Retriever invoke returns the top_k docs
keyword_retriever.invoke(query) 

[Document(metadata={'source': '../data/processed/preprocessed_corpus.csv', 'row': 1957, 'asin': 'B000Y0CL8K', 'product_title': 'Travelon Hand Soap Toiletry Sheets, 50-Count', 'rating': '4.0', 'review_text': 'A single sheet is good for a pair of underwear. The soap dissolves quickly but the water does not feel too soapy so not sure about the quality of cleaning. Best is to let the clothes sit in the soap solution for 5 minutes before rinsing.'}, page_content='text: hand soap toiletry sheets count health personal care lightweight soap travel single sheet good pair underwear soap dissolve water feel soapy sure quality cleaning good let clothe sit soap solution minute rinse'),
 Document(metadata={'source': '../data/processed/preprocessed_corpus.csv', 'row': 2608, 'asin': 'B000Y0CL8K', 'product_title': 'Travelon Hand Soap Toiletry Sheets, 50-Count', 'rating': '3.0', 'review_text': "If you MUST save on space, consider these laundry soap sheets.  I will be honest, though... hotel bar soap and

## Ensemble BM25 + Semantic Retriever

In [8]:
# Invoke to get combined results
ensemble_retriever = hybrid_retriever(keyword_retriever, vector_retriever)
ensemble_retriever.invoke(query)

[Document(metadata={'source': '../data/processed/preprocessed_corpus.csv', 'row': 1957, 'asin': 'B000Y0CL8K', 'product_title': 'Travelon Hand Soap Toiletry Sheets, 50-Count', 'rating': '4.0', 'review_text': 'A single sheet is good for a pair of underwear. The soap dissolves quickly but the water does not feel too soapy so not sure about the quality of cleaning. Best is to let the clothes sit in the soap solution for 5 minutes before rinsing.'}, page_content='text: hand soap toiletry sheets count health personal care lightweight soap travel single sheet good pair underwear soap dissolve water feel soapy sure quality cleaning good let clothe sit soap solution minute rinse'),
 Document(id='6ddf5f18-1ad1-4985-bb3f-1da5a117cb64', metadata={'source': 'data/processed/preprocessed_corpus.csv', 'row': 1761, 'asin': 'B001CGOPZM', 'product_title': 'Travelon Hand Soap Toiletry Sheets, 50-Count', 'rating': '5.0', 'review_text': 'nice product'}, page_content='text: hand soap toiletry sheets count he

In [9]:
rag_chain = initialize_rag_chain(ensemble_retriever, llm, prompt)
print(invoke_rag_chain(rag_chain, query))

Based on the reviews, here are some good soaps mentioned:

1. Dial Mountain Fresh Antibacterial Deodorant Soap, 4 oz, 6 Count (B00OPC61V6) - This soap is praised for its ability to fight odor, leaving the user feeling clean and fresh. It's also mentioned that it's hard to find, but really liked for its fragrance.
2. Sekkisei cream wash (B004UJHY74) - This soap is described as smelling nice and soapy, with a creamy texture that does a great job. It's mentioned as a regular purchase for one reviewer.
3. Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool (Not a soap itself, but a useful tool for keeping soap bars clean and fresh) (B0716PQVP2)

Please note that the Travelon Hand Soap Toiletry Sheets (B000Y0CL8K and B001CGOPZM) received mixed reviews, with some users praising their convenience and ease of use, but others finding them not as effective at cleaning or lasting as long.
